# 🧠 RMSProp Optimization: Adaptive Coordinates

Welcome to the hands-on explanation notebook for **RMSProp Optimization**! In this notebook, we will:
1. Explain the limitations of AdaGrad (vanishing learning rate) and how RMSProp's exponentially decaying average solves it.
2. Implement **Vanilla Gradient Descent**, **AdaGrad**, and **RMSProp** from scratch.
3. Track and compare their optimization trajectories on our steep 2D ravine cost function:
   $$f(x, y) = 0.5x^2 + 10y^2$$
4. Visualize their paths on a 2D contour map to observe how AdaGrad freezes early and how RMSProp dynamically adapts coordinate learning rates.
5. Plot loss curves to contrast convergence.
6. Connect adaptive learning rates to deep neural network training (such as training YOLO model layers).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Defining the Ravine Function and Gradients

Our cost function represents a steep valley:
$$f(x, y) = 0.5x^2 + 10y^2$$

Gradients:
$$\frac{\partial f}{\partial x} = x, \quad \frac{\partial f}{\partial y} = 20y$$

In [ ]:
def cost_ravine(x, y):
    return 0.5 * x**2 + 10.0 * y**2

def grad_ravine(x, y):
    return np.array([x, 20.0 * y])

## 2. Implementing Optimizers from Scratch

Let's write three optimization loops:
1.  **Vanilla GD:** $\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \nabla J(\mathbf{w}_t)$
2.  **AdaGrad:** Cumulative sum of squared gradients in the denominator.
3.  **RMSProp:** Exponentially decaying running average of squared gradients in the denominator.

In [ ]:
def optimize_vanilla(start_pos, lr=0.15, epochs=60):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_adagrad(start_pos, lr=0.5, eps=1e-8, epochs=60):
    pos = np.array(start_pos, dtype=float)
    s = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        s += grad ** 2
        pos -= (lr / (np.sqrt(s) + eps)) * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_rmsprop(start_pos, lr=0.15, beta=0.9, eps=1e-8, epochs=60):
    pos = np.array(start_pos, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        v = beta * v + (1.0 - beta) * (grad ** 2)
        pos -= (lr / (np.sqrt(v) + eps)) * grad
        history.append(pos.copy())
    return np.array(history)

# Run optimizations starting at (8.0, 4.0)
start = [8.0, 4.0]
path_vanilla = optimize_vanilla(start, lr=0.08)
path_adagrad = optimize_adagrad(start, lr=0.8)
path_rmsprop = optimize_rmsprop(start, lr=0.15, beta=0.9)

## 3. Visualizing Trajectories over the Contour Map

Let's generate the 2D contour grid and plot the paths.

In [ ]:
x = np.linspace(-10, 10, 150)
y = np.linspace(-5, 5, 150)
X, Y = np.meshgrid(x, y)
Z = cost_ravine(X, Y)

plt.figure(figsize=(12, 8))
contours = plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

# Plot paths
plt.plot(path_vanilla[:, 0], path_vanilla[:, 1], color='red', marker='o', alpha=0.8, linewidth=1.5, label='Vanilla GD')
plt.plot(path_adagrad[:, 0], path_adagrad[:, 1], color='purple', marker='^', linewidth=2, label='AdaGrad (Stalls early)')
plt.plot(path_rmsprop[:, 0], path_rmsprop[:, 1], color='cyan', marker='s', linewidth=2.5, label='RMSProp (Glides to minimum)')

plt.scatter(0, 0, color='gold', s=150, marker='*', zorder=5, label='Minimum (0,0)')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-10, 10)
plt.ylim(-5, 5)
plt.title('AdaGrad Stalling vs. RMSProp Adaptive Adjustments')
plt.legend()
plt.show()

Look at the plot!
-   **Vanilla GD (Red):** Oscillates wildly back-and-forths because of the steep y-axis gradient, progressing very slowly toward $x=0$.
-   **AdaGrad (Purple):** Because the initial gradients are huge, the historical sum $s$ blows up, dividing the learning rate to near-zero. It stops moving (stalls) far away from the global minimum.
-   **RMSProp (Cyan):** By decay-averaging the squared gradients, it prevents the denominator from exploding. It dampens the vertical $y$-oscillations while scaling up the horizontal $x$-updates, gliding directly to the minimum.

## 4. Comparing Convergence Speeds

Let's plot the cost reduction curves.

In [ ]:
cost_vanilla = [cost_ravine(p[0], p[1]) for p in path_vanilla]
cost_adagrad = [cost_ravine(p[0], p[1]) for p in path_adagrad]
cost_rmsprop = [cost_ravine(p[0], p[1]) for p in path_rmsprop]

plt.figure(figsize=(10, 5))
plt.plot(cost_vanilla, color='red', label='Vanilla GD')
plt.plot(cost_adagrad, color='purple', label='AdaGrad')
plt.plot(cost_rmsprop, color='cyan', label='RMSProp')
plt.yscale('log')
plt.xlabel('Steps')
plt.ylabel('Log Cost')
plt.title('Cost Convergence Comparison (Log Scale)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 💡 Connection to YOLO and Deep Learning
*   **Vanishing Gradients across layers:** In deep networks (like YOLO's CNN backbone), initial convolutional layers are far from the output loss, so their gradients are extremely small. In contrast, the final classification and regression heads have large gradients.
*   **Adaptive Scaling:** If we used a single global learning rate, we would either overshoot the final layer parameters or fail to update the initial layer parameters. Adaptive optimizers like RMSProp (and its derivative, Adam) scale parameters independently, allowing each layer to learn at its own optimal speed!